# Setup
This Nodebook should help generate the input files used for the mammalian cell culture simulation. 

In [ ]:
from pathlib import Path

from dbiolab.mammalian_cellculture_simulation import (
    CellType,
    FeedConcentrations,
    FeedingEvent,
    MammalianCellcultureReactor,
    MammalianRecipe,
    NoiseParameters,
    ReactorType,
    plot_reaction_evolution_timeseries,
)

LOCAL_DIR = "../.local"
Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
filepaths = {
    "reactor": f"{LOCAL_DIR}/MTP_1.json",
    "celltype": f"{LOCAL_DIR}/celltype_1_alpha.json",
    "initial_conditions": f"{LOCAL_DIR}/initial_conditions.json",
    "feed_event": f"{LOCAL_DIR}/feed_event.json",
    "noise": f"{LOCAL_DIR}/noise_levels.json",
}

## 1. Define Reactor Type

In [10]:
reactor = ReactorType(
    name="MTP_1",
    volume=3.0,
    scale_factor=5.0,
    growth_inhibition=0.95,
).to_json(filepaths["reactor"])

## 2. Define Cell Type

In [11]:
celltype = CellType(
    name="celltype_1_alpha",
    mu_max=0.0451639932,
    k_lysis=0.0446990515,
    k_deg_gln=0.0013441722,
    k_death_max=0.0101556537,
    k_death_mu=0.0145125692,
    k_sat_lac=141.4574131165,
    k_sat_ammonia=37.778062819,
    k_sat_glc=1.1634941712,
    k_sat_gln=0.2123961928,
    y_ammonia=9.98e-08,
    y_lactate=0.8061528449,
    y_cells_glc=153303245.24868935,
    y_cells_gln=8363679425.5848055,
    m_glc=4e-10,
    m_gln=0.0,
    ph_opt=6.6,
    ph_opt_product=6.6,
    ph_robustness_factor=1.0,
    ph_opt_product_robustness=0.8,
    e_activation=37.4,
    temperature_opt_product=37.4,
    temperature_robustness_factor=1.0,
    temperature_opt_product_robustness=0.8,
    y_product_from_total=9.98e-08,
    y_product_from_viable=4.9308e-06,
    hcp_per_cell=1e-6,
    k_agg_product=1e-1,
).to_json(filepaths["celltype"])

## 3. Define Recipe

### Initial Conditions

In [12]:
initial_conditions = FeedConcentrations(
    c_product=0.0,
    x_total=800000.0,
    x_viable=800000.0,
    x_dead=0.0,
    c_glc=4.0,
    c_gln=4.0,
    c_lac=0.0,
    c_ammonia=0.0,
    c_aggregates=0.0,
).to_json(filepaths["initial_conditions"])

### Feed Event

In [13]:
feed_event = FeedingEvent(
    time_point=700.0,
    feed_concentrations=FeedConcentrations(
        c_product=0.0,
        x_total=0.0,
        x_viable=0.0,
        x_dead=0.0,
        c_glc=2.0,
        c_gln=0.0,
        c_lac=0.0,
        c_ammonia=0.0,
        c_aggregates=0.0,
    ),
).to_json(filepaths["feed_event"])

### Noise in Simulations

In [14]:
NoiseParameters(
    process_noise_c_product=0.1,
    process_noise_x_total=0.1,
    process_noise_x_viable=0.1,
    process_noise_x_dead=0.1,
    process_noise_c_glc=0.1,
    process_noise_c_gln=0.1,
    process_noise_c_lac=0.1,
    process_noise_c_ammonia=0.1,
    process_noise_c_aggregates=0.1,
    measurement_noise_c_product=0.05,
    measurement_noise_x_total=0.05,
    measurement_noise_x_viable=0.05,
    measurement_noise_x_dead=0.05,
    measurement_noise_c_glc=0.05,
    measurement_noise_c_gln=0.05,
    measurement_noise_c_lac=0.05,
    measurement_noise_c_ammonia=0.05,
    measurement_noise_c_aggregates=0.05,
).to_json(filepaths["noise"])

## 4. Run Simulation

In [15]:
reactor = ReactorType.from_json(filepaths["reactor"])
simulator = MammalianCellcultureReactor(reactor_type=reactor)
celltype = CellType.from_json(filepaths["celltype"])
initial_conditions = FeedConcentrations.from_json(filepaths["initial_conditions"])
feed_event = FeedingEvent.from_json(filepaths["feed_event"])
noise_parameters = NoiseParameters.from_json(filepaths["noise"])
feeding_strategy = [feed_event]
recipe = MammalianRecipe(
    name="Test Recipe",
    celltype=celltype,
    temperature=37.4,  # range 30-40
    ph_value=6.6,  # range 6-8
    initial_conditions=initial_conditions,
    feeding_strategy=feeding_strategy,
    runtime=1336.0,
    measurement_intervals=12.0,
    noise_parameters=noise_parameters,
)
solution = simulator.run_experiment(recipe=recipe)

## 5. Visualize Results

In [16]:
solution_plots = plot_reaction_evolution_timeseries(solution)
for fig in solution_plots:
    fig.show()